<a href="https://colab.research.google.com/github/SujaAK/HindiASR/blob/main/MMS_TTS_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers accelerate -q
!pip install datasets huggingface_hub soundfile -q
!pip install openai-whisper jiwer -q

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

llm_name = "unsloth/Llama-3.2-3B-Instruct"

print(f"Loading {llm_name} ...")
llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("LLM loaded successfully!")

In [ ]:
import re

DOMAIN_PROMPTS = {
    "dairy_order": "एक ग्राहक डेयरी की दुकान को फोन पर दूध, दही, पनीर या घी का ऑर्डर दे रहा है।",
    "payment": "एक ग्राहक या दुकानदार पेमेंट, बिल, चालान या क्रेडिट नोट के बारे में बात कर रहा है।",
    "delivery": "डिलीवरी का समय, लोकेशन या टाइमिंग बदलने को लेकर बातचीत हो रही है।",
    "order": "नया ऑर्डर देना, ऑर्डर कैंसिल करना या रिपीट ऑर्डर को लेकर बातचीत हो रही है।",
    "general": "स्टॉक, रेट, क्वालिटी कंप्लेंट या दुकान बंद होने को लेकर सामान्य बातचीत हो रही है।",
}

SYSTEM_PROMPT = (
    "आप एक भारतीय छोटे व्यापारी (दुकानदार) और उसके ग्राहक के बीच होने वाली स्वाभाविक, "
    "बोलचाल की फोन कॉल बातचीत का एक वाक्य लिखते हैं। हमेशा देवनागरी लिपि में लिखें। "
    "अंग्रेज़ी के शब्द भी देवनागरी में लिखें (जैसे 'पेमेंट', 'डिलीवरी'), कभी भी लैटिन/रोमन "
    "लिपि का उपयोग न करें। संख्याएं हमेशा शब्दों में लिखें, अंकों में नहीं (जैसे 'दस' न कि '10')। "
    "सिर्फ एक छोटा, प्राकृतिक वाक्य दें, कोई अनुवाद या स्पष्टीकरण नहीं।"
)

hinglish_markers = ["पेमेंट", "डिलीवरी", "कन्फर्म", "स्टॉक", "क्वालिटी", "कंप्लेंट",
                     "चालान", "क्रेडिट", "कैंसिल", "रिपीट", "अवेलेबल"]


def is_devanagari_only(text):
    """True if the sentence has no Latin alphabet characters (a-z, A-Z)."""
    return not re.search(r'[a-zA-Z]', text)


def tag_language_mix(sentence):
    return "hinglish" if any(m in sentence for m in hinglish_markers) else "hindi"


def generate_llama_sentence(domain, max_new_tokens=80, temperature=0.9):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"विषय: {DOMAIN_PROMPTS[domain]}\nएक वाक्य लिखें:"}
    ]

    # return_dict=False ensures we get a plain tensor (with .shape), not a
    # BatchEncoding dict-like object - fixes the KeyError: 'shape' error.
    input_ids = llm_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False
    ).to(llm_model.device)

    with torch.no_grad():
        output = llm_model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.92,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    generated = llm_tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    sentence = generated.strip().split("\n")[0].strip()
    sentence = sentence.strip('"\'""\'\'')
    sentence = re.sub(r'\d+', '', sentence).strip()
    return sentence


def generate_llama_batch(target_count=30, domains=None):
    domains = domains or list(DOMAIN_PROMPTS.keys())
    per_domain_target = max(1, target_count // len(domains))
    domain_counts = {d: 0 for d in domains}
    records, seen = [], set()
    attempts, max_attempts = 0, target_count * 6

    while len(records) < target_count and attempts < max_attempts:
        attempts += 1
        under = [d for d in domains if domain_counts[d] < per_domain_target]
        domain = under[attempts % len(under)] if under else domains[attempts % len(domains)]

        sentence = generate_llama_sentence(domain)

        if not sentence or sentence in seen:
            continue
        if not is_devanagari_only(sentence):
            continue
        if len(sentence.split()) < 3 or len(sentence.split()) > 25:
            continue

        seen.add(sentence)
        domain_counts[domain] += 1
        records.append({
            "transcript": sentence,
            "domain_tag": domain,
            "language_mix": tag_language_mix(sentence),
            "sentence_length": len(sentence.split()),
            "source": "llama"
        })

        if len(records) % 10 == 0:
            print(f"  Generated {len(records)}/{target_count}...")

    return records


# Test with a small batch first (30 sentences) before scaling up
llama_dataset = generate_llama_batch(target_count=30)
print(f"\nTotal generated: {len(llama_dataset)}")
for r in llama_dataset[:10]:
    print(f"[{r['domain_tag']:12s}] {r['transcript']}")

In [ ]:
from transformers import VitsModel, AutoTokenizer as TTSTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

tts_model = VitsModel.from_pretrained("facebook/mms-tts-hin").to(device)
tts_tokenizer = TTSTokenizer.from_pretrained("facebook/mms-tts-hin")

print("MMS-TTS-Hindi loaded successfully!")

In [ ]:
import os
import soundfile as sf

os.makedirs("generated_audio", exist_ok=True)
audio_records = []
failed_samples = []

for idx, record in enumerate(llama_dataset):
    text = record["transcript"]
    try:
        inputs = tts_tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            output = tts_model(**inputs).waveform
        audio_arr = output.cpu().numpy().squeeze()

        filename = f"generated_audio/sample_{idx:05d}.wav"
        sf.write(filename, audio_arr, tts_model.config.sampling_rate)

        audio_records.append({
            "audio": filename,
            "transcript": text,
            "domain_tag": record["domain_tag"],
            "language_mix": record["language_mix"],
            "sentence_length": record["sentence_length"]
        })
    except Exception as e:
        print(f"FAILED sample {idx}: '{text}' — {e}")
        failed_samples.append((idx, text, str(e)))
        continue

print(f"\nDone! {len(audio_records)} succeeded, {len(failed_samples)} failed")

In [ ]:
import whisper
import jiwer

whisper_model = whisper.load_model("base")

predictions, references = [], []

for record in audio_records:
    result = whisper_model.transcribe(
        record["audio"],
        language="hi",
        task="transcribe",
        initial_prompt="यह हिंदी में बातचीत है।"  # nudges toward Devanagari/Hindi context
    )
    predictions.append(result["text"])
    references.append(record["transcript"])

wer = jiwer.wer(references, predictions)
cer = jiwer.cer(references, predictions)

print(f"WER: {wer:.3f}, CER: {cer:.3f}")

for ref, pred in zip(references[:10], predictions[:10]):
    print(f"GT:   {ref}")
    print(f"Pred: {pred}\n")